### Zadanie 1 · PostgreSQL (lub SQLite)

Korzystając z **Random User API** (`https://randomuser.me/api/?results=30`), wygeneruj 30 użytkowników i zapisz ich dane (imię, nazwisko, email, adres, wiek, płeć) w tabeli SQL.

Następnie sprawdź SQL-em:
- Ile jest mężczyzn, a ile kobiet?
- Jaki jest średni wiek?
- W ilu krajach mieszkają?

*Tip: `requests.get(...).json()["results"]` zwraca listę dictów — iteruj i INSERT-uj po jednym.*  
*Możesz użyć SQLite (jak w tym notebooku) lub PostgreSQL jeśli masz zainstalowany.*

In [3]:
# Zadanie 1 -- Twoj kod tutaj
import sqlite3
import requests

# 1. Pobierz dane z API
print("Pobieranie danych z API...")
response = requests.get("https://randomuser.me/api/?results=30")
users = response.json()["results"]

# Połączenie z bazą danych (':memory:' tworzy tymczasową bazę w pamięci RAM)
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# 2. Stwórz tabelę Users (id, first_name, last_name, email, age, gender, country)
cursor.execute('''
    CREATE TABLE IF NOT EXISTS Users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        first_name TEXT,
        last_name TEXT,
        email TEXT,
        age INTEGER,
        gender TEXT,
        country TEXT
    )
''')

# 3. Wstaw dane z parametryzacją (zapobiega SQL Injection)
insert_query = '''
    INSERT INTO Users (first_name, last_name, email, age, gender, country)
    VALUES (?, ?, ?, ?, ?, ?)
'''

for user in users:
    # Wyciąganie odpowiednich wartości ze słownika (odpowiedzi JSON z API)
    first_name = user['name']['first']
    last_name = user['name']['last']
    email = user['email']
    age = user['dob']['age']
    gender = user['gender']
    country = user['location']['country'] # używamy kraju jako adresu (najlepsze do analityki)
    
    # Przekazanie krotki (tuple) z danymi do zapytania parametryzowanego
    cursor.execute(insert_query, (first_name, last_name, email, age, gender, country))

# Zatwierdzenie zmian (INSERTów) w bazie
conn.commit()

# 4. Zapytania analityczne

print("\n--- Ile jest mężczyzn, a ile kobiet? ---")
cursor.execute("SELECT gender, COUNT(*) FROM Users GROUP BY gender")
for row in cursor.fetchall():
    print(f"Płeć {row[0]}: {row[1]}")

print("\n--- Jaki jest średni wiek? ---")
cursor.execute("SELECT AVG(age) FROM Users")
avg_age = cursor.fetchone()[0]
print(f"Średni wiek użytkowników to: {avg_age:.2f} lat")

print("\n--- W ilu krajach mieszkają? (Liczba unikalnych krajów) ---")
cursor.execute("SELECT COUNT(DISTINCT country) FROM Users")
country_count = cursor.fetchone()[0]
print(f"Użytkownicy mieszkają w {country_count} różnych krajach.")

print("\n--- Lista krajów i liczba mieszkańców ---")
cursor.execute("SELECT country, COUNT(*) FROM Users GROUP BY country ORDER BY COUNT(*) DESC")
for row in cursor.fetchall():
    print(f"Kraj: {row[0]}, Liczba osób: {row[1]}")

conn.close()

Pobieranie danych z API...

--- Ile jest mężczyzn, a ile kobiet? ---
Płeć female: 14
Płeć male: 16

--- Jaki jest średni wiek? ---
Średni wiek użytkowników to: 55.43 lat

--- W ilu krajach mieszkają? (Liczba unikalnych krajów) ---
Użytkownicy mieszkają w 17 różnych krajach.

--- Lista krajów i liczba mieszkańców ---
Kraj: United States, Liczba osób: 4
Kraj: Mexico, Liczba osób: 4
Kraj: United Kingdom, Liczba osób: 3
Kraj: Norway, Liczba osób: 3
Kraj: Switzerland, Liczba osób: 2
Kraj: Denmark, Liczba osób: 2
Kraj: Canada, Liczba osób: 2
Kraj: Ukraine, Liczba osób: 1
Kraj: Turkey, Liczba osób: 1
Kraj: Serbia, Liczba osób: 1
Kraj: New Zealand, Liczba osób: 1
Kraj: Netherlands, Liczba osób: 1
Kraj: Ireland, Liczba osób: 1
Kraj: India, Liczba osób: 1
Kraj: Germany, Liczba osób: 1
Kraj: France, Liczba osób: 1
Kraj: Finland, Liczba osób: 1


### Zadanie 2 · MongoDB

Z API GeckoTerminal (`https://api.geckoterminal.com/api/v2/networks`) pobierz informacje o dostępnych sieciach kryptowalutowych. Zapisz je jako dokumenty w kolekcji MongoDB.

Użyj agregacji, żeby policzyć liczbę sieci per typ.

*Tip: MongoDB akceptuje dict bezpośrednio — żadnej parametryzacji jak w SQL.*

**Wymaga działającego serwera MongoDB** (lokalnie, Docker lub Atlas).

In [4]:
# Zadanie 2 -- Twoj kod tutaj
# from pymongo import MongoClient
from pymongo import MongoClient
import requests

print("1. Łączenie z bazą MongoDB...")
client = MongoClient("mongodb://localhost:27017")
db = client.lab4
networks = db["networks"]

# Czyścimy kolekcję przed ponownym pobraniem, by uniknąć duplikatów przy wielokrotnym uruchamianiu
networks.delete_many({})

# API
print("2. Pobieranie danych z API GeckoTerminal...")
response = requests.get("https://api.geckoterminal.com/api/v2/networks")
data = response.json()["data"]

print(f"Pobrano {len(data)} sieci. Wstawianie do MongoDB...")
# 3. inserty

networks.insert_many(data)

print("\n4. Wynik agregacji (Liczba sieci per typ):")
# 4. Agregacja 
pipeline = [
    # Krok 1: Grupujemy
    {"$group": {
        "_id": "$type", 
        "count": {"$sum": 1}
    }},
    # Krok 2: Sortujemy malejąco po wyliczonym polu "count"
    {"$sort": {"count": -1}}
]

# Pipeline i select wyników
for doc in networks.aggregate(pipeline):
    print(f"Typ sieci: '{doc['_id']}' | Liczba: {doc['count']}")

# Zamknięcie połączenia dla pewności
client.close()

1. Łączenie z bazą MongoDB...
2. Pobieranie danych z API GeckoTerminal...
Pobrano 100 sieci. Wstawianie do MongoDB...

4. Wynik agregacji (Liczba sieci per typ):
Typ sieci: 'network' | Liczba: 100
